In [ ]:
# Note:    ipykernel    must first be installed (do    pip install ipykernel    ) in the current Python environment in order to run this Jupyter notebook. The Jupyter notebook interface relies on this package to execute Python code within the notebook cells.

# Create the virtual environment:    cd    into the root folder and run    python3 -m venv venv
# Activate the virtual environment in the terminal: still in the root folder and run    source venv/bin/activate
# Install required packages (if not already installed):    pip install ipykernel torch matplotlib numpy umap-learn
# Add your virtual environment,    venv    , as a Jupyter kernel by running the following in terminal:    python -m ipykernel install --user --name=myenv --display-name "Python (myenv)"
# From the topright corner of VS Code select the kernel named "venv (1.2.3) (Python 1.2.3)" in the notebook interface, where 1.2.3 is the version of Python in your virtual environment.

# Import all required libraries.

import torch # Import PyTorch.
import torch.nn.functional as F # Import the functional API as F in order to use    cross_entropy    and    softmax    functions without building    nn.Module    objects. This approach accesses PyTorch's core neural network operations directly, without the convenient abstraction of model classes. While it offers flexibility and transparency, making it easier to see and control every step of the computation, it also requires manually creating and updating all model parameters, manually handling forward passes, and manually managing gradients, which can be prone to error and less scalable for larger projects. Getting this close to the metal is great for learning and experimentation, or whenever fine grained control is needed.
import matplotlib.pyplot as plt # Import Matplotlib in order to visualize plotting the loss curve, specifying the "magic"    %matplotlib inline    command to enable inline Matplotlib rendering in Jupyter/Colab environments so plots show as a png in the output cells. Note,    %matplotlib inline    is already the default, however explicitly using it is still common for clarity and compatibility.
%matplotlib inline
import numpy as np # Import NumPy for occasional numeric utilities. Note, in the original Google Colab from where this was inspired, Karpathy used    .data    for plotting (e.g.,    plt.scatter(C[:,0].data, ...)    ), but that implementation was not portable outside of Google Colab and had to be exchanged for    .detach().numpy()    , in which    .detach()    is necessary to get the raw tensor values without tracking gradients, and    .numpy()    converts the PyTorch tensor to a NumPy array for plotting with Matplotlib, because Matplotlib does not understand PyTorch tensors directly.
import umap # Import UMAP for dimensionality reduction of the character embeddings, which can be used to visualize high-dimensional data in 2D or 3D space while preserving local structure. This is particularly useful for visualizing the learned high-dimensional character embeddings, for which only the first two dimensions would otherwise be able to be displayed.
import os # Import os to be able to check for the existence of and load any saved model file before attempting to initialize model parameters from scratch.
import random # Import Python's    random    module for shuffling/splitting of the raw list, which can be used deterministically with a fixed seed for reproducibility.

In [ ]:
# Establish the data.

words = open('data/englishWordsList.txt', 'r').read().splitlines() # Chain the file opening, reading, and splitting into individual lines in a single step to create the list of words. This is a common Python idiom for concise file processing.

parsing_character = '.' # Define a parsing character to represent the end of a word.

print(f'The first 7 items are { words[:7] } and the length of the list is { len(words) }.') # This is simply to verify that the file was read correctly and the list of words was created as expected.

In [ ]:
# Build the mappings.

chars = sorted(list(set(''.join(words)))) # First joining all the words together into a massive string, create a set from it, thereby effectively removing duplicates of all characters, then convert that set to a list in order to be able to perform list specific operations such as indexing, and then sort that list alphabetically.

string_to_index = { s: i+1 for i, s in enumerate(chars) } # Enumerating the characters into a raw memory-less iterator that returns tuples of (index, character), build a dictionary that maps character to index + 1, intentionally reserving the 0 index for the parsing character...
string_to_index[parsing_character] = 0 # ...and finally tack on the parsing character, giving it index 0.

index_to_string = { i: s for s, i in string_to_index.items() } # Breaking this    string_to_index    dictionary into individual key-value pairs, for each index, map it back to its respective string in a new dictionary called    index_to_string    .

print(f'The index_to_string dictionary is: {index_to_string}') # This is simply to verify that the    index_to_string    ...
print(f'The string_to_index dictionary is: {string_to_index}') # ...and    string_to_index    dictionaries were built correctly.

In [ ]:
# Initialize the datasets for training, validation, and testing.

context_length = 4 # Define the    context_length    , which is simply how many consecutive tokens, which in this case are characters, the neural net should receive in order to predict the next one.


def initialize_dataset(words, type): # Define a function to initialize an integer based dataset. It takes in a list of words and returns two tensors,    X    and    Y    , where X is a list of input contexts (each context is a list of integers representing previous characters of length    context_length    ), and Y is a list of target characters (integers representing the next character to predict). The    type    enum is simply for printing purposes below.

    X, Y = [], []

    for word in words:

        context = [ string_to_index[parsing_character] ] * context_length # Initialize the context list of repeating indices corresponding to the parsing character, with the length of the list equal to the    context_length    . This effectively creates a list containing a sequence of parsing character indices, namely a sequence of zeros, that represent the start of a word.

        for ch in word + parsing_character: # Loop through each character in the word plus the parsing_character...

            ix = string_to_index[ch] # ...where the index of the current character is first looked up in the    string_to_index    dictionary...

            X.append(context) # ...the context list of indices that comes before the character is appended to the list of input contexts...
            Y.append(ix) # ...the index of the current character is appended to the list of target characters...

            #print(''.join(index_to_string[i] for i in context), '--->', index_to_string[ix])

            context = context[1:] + [ix] # ...and then crop the context by removing the first element, and append the current character index to the end of the list, effectively sliding the context window forward by one character to prepare it for the next iteration.

    # Transform the    X    contexts and    Y    targets lists into tensors.
    X = torch.tensor(X)
    Y = torch.tensor(Y)

    print(f'The {type} input tensor has a shape of {X.shape} and output tensor has a shape of {Y.shape}') # This is simply to verify that the tensors were created correctly and have the expected shapes, in which case the lists of contexts and targets must have also been created correctly. These shapes are compatible with PyTorch's broadcasting rules if they meet the criteria specified in    https://docs.pytorch.org/docs/stable/notes/broadcasting.html#general-semantics    . The input tensor should be 2D with the shape (number of training examples, context_length) and the output tensor should be 1D with the shape (number of training examples).

    return X, Y # Return the input and target tensors as two separate objects.

random_python_seed = 123 # Define an arbitrary seed to pass to Python's built-in    random    module. This is purely to be able to reproduce the same values when rerunning the code, effectively and ironically making the    random    module nonrandom, but which is important for debugging and consistency of results. Without setting seed, the shuffling and splitting of the dataset would yield different results each time the code is run. This could technically be any number.

random.seed(random_python_seed) # Set the seed for deterministic reproducibility of the shuffling and splitting of the dataset...
random.shuffle(words) # ...and shuffle the list of words in place also using Python's built-in    random    module, which is a common practice to ensure that the training, validation, and test sets are representative of the overall dataset and not biased by any particular ordering of the data. Note, that "set" in this context does not refer to the data structure, but rather to the training, validation, and test lists. "Set" is simply the common term for this in machine learning.

dataset_split_index_train = int(0.8*len(words)) # Define the index at which to begin splitting the dataset into training and validation sets.
dataset_split_index_validate = int(0.9*len(words)) # Define the index at which to begin splitting the dataset into validation and test sets.

X_train, Y_train = initialize_dataset(words[ :dataset_split_index_train], 'Training' ) # Build the training dataset by passing    words    up to    dataset_split_index_train.
X_validate, Y_validate = initialize_dataset(words[ dataset_split_index_train:dataset_split_index_validate ], 'Validation' ) # Build the validation dataset by passing    words    from    dataset_split_index_train    to    dataset_split_index_validate    .
X_test, Y_test = initialize_dataset(words[ dataset_split_index_validate:], 'Testing' ) # Build the testing dataset by passing the remaining    words    from    dataset_split_index_validate    to the end.

In [ ]:
# Initialize the model.

vocab_size = len(string_to_index)  # This is the number of unique tokens in the vocabulary which, for this model, is the number of unique characters in the data plus the parsing character. Note, because this is a character-level language model, the vocabulary in this case is simply characters, not words.

embedding_dimensions = 13 # This is the number of dimensions in which to embed each character, which serve as distributed representations of the vocabulary capturing "features" that wind up abstractly representing semantic relationships among the vocabulary. This was the breakthrough in Bengio's seminal 2003 paper "A Neural Probabilistic Language Model"    https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf    . "Vocabulary" refers to the set of unique tokens in the dataset which, in this case, is simply characters. Note, although each embedding is a 1D vector, it is treated as a high-dimensional scalar vector, where "dimensions" refers to the number of scalar values, or "features", that represent each character in the embedding space. If each scalar were to be visualized it would exist as a single point in that n-dimensional space.

concatenated_context_length = context_length * embedding_dimensions # This is the length of the concatenated context vector that will be fed into the input layer of the multi-layer perceptron (MLP). It is simply each of the context characters' n-dimensional embeddings concatenated together.

num_of_neurons_in_hidden_layer = 200 # This is the number of neurons in the hidden layer of the MLP, which is an arbitrary hyperparameter that can be tuned for better performance.

random_pytorch_seed = 123 # Define an arbitrary seed to pass to PyTorch's random number generator. This is purely to be able to reproduce the same values when rerunning the code, effectively and ironically making the random number generator nonrandom, but which is important for debugging and consistency of results. Without setting a seed, the initialization of the model parameters would yield different results each time the code is run. This could technically be any number.
g = torch.Generator().manual_seed(random_pytorch_seed)


if os.path.exists('char_level-embed_mlp_model.pt'): # Check if a saved model file exists in the current directory with the name 'char_level-embed_mlp_model.pt'. If it does, load the model parameters from that file instead of initializing them from scratch. This allows for resuming training from a previous checkpoint or using a pre-trained model for inference without having to retrain it from the beginning.

    training_checkpoint = torch.load('char_level-embed_mlp_model.pt')

    C = training_checkpoint['C']
    W1 = training_checkpoint['W1']
    b1 = training_checkpoint['b1']
    batch_norm_gain_1 = training_checkpoint['batch_norm_gain_1']
    batch_norm_bias_1 = training_checkpoint['batch_norm_bias_1']
    batch_norm_mean_running_1 = training_checkpoint['batch_norm_mean_running_1']
    batch_norm_standard_deviation_running_1 = training_checkpoint['batch_norm_standard_deviation_running_1']
    batch_norm_epsilon = training_checkpoint['batch_norm_epsilon']
    batch_norm_momentum = training_checkpoint['batch_norm_momentum']
    W2 = training_checkpoint['W2']
    b2 = training_checkpoint['b2']
    
    parameter_tensors = [C, W1, b1, batch_norm_gain_1, batch_norm_bias_1, W2, b2]  # This is a list of the 7 top-level learnable tensors (C, W1, b1, batch_norm_gain_1, batch_norm_bias_1, W2, b2) that compose the model.

    print('Number of parameters in loaded model: ')

else:

    C = torch.randn((vocab_size, embedding_dimensions), generator=g) # This is the randomly initialized (drawn from a standard normal, i.e. Gaussian, distribution) character embedding matrix, which is a learnable parameter tensor of shape (vocab_size, embedding_dimensions). Each row corresponds to a unique character in the vocabulary, and each column corresponds to a dimension in the embedding space. The values in this matrix are initialized randomly from a normal distribution using the specified random seed for reproducibility. During training, the model will learn to adjust these values such that characters with similar "features" in the training data have similar embeddings, allowing the model to capture semantic relationships between characters. The name for this matrix,    C    , doesn't have anything to do with "characters", its purely a convention from Bengio's original paper, the naming of which he never explained.

    W1 = torch.randn((concatenated_context_length, num_of_neurons_in_hidden_layer), generator=g) * (5/3 * (1/concatenated_context_length)**0.5) # This is the randomly initialized (drawn from a standard normal, i.e. Gaussian, distribution) first weight matrix which, in this case, is used to train the hidden layer. Its first dimension is the length of the concatenated input vector of "features" (context_length × embedding_dimensions), and its second dimension is the number of neurons in the hidden layer. Because the initialization of    W1    shouldn't have too opinionated a value and the distribution that    torch.randn    would give would indeed be that, i.e. it could have a wide range, it's shrewd to reel it in with a coefficient to narrow the initialization. The manner in which to do this optimally and dynamically is with Kaiming initialization, aka He initialization, the equation for which is    gain * sqrt(1/fan_in)    where    fan_in    is the number of input units in the weight tensor, which in this case is the concatenated_context_length, and    gain    is a scaling factor set according to the nonlinearity in question. For the tanh nonlinearity the approximately optimal gain is 5/3, which has been derived roughly and empirically, and has been generally agreed upon heuristically by the machine learning community, i.e. there's no single paper that lists a formal symbolic proof for exactly why 5/3 is the gain used for tanh.
    b1 = torch.randn(num_of_neurons_in_hidden_layer, generator=g) * .01 # This is the randomly initialized (drawn from a standard normal, i.e. Gaussian, distribution) bias vector for the hidden layer, which is added to the output of the linear transformation with    W1    before nonlinearity is applied. Its length is accordingly equal to the number of neurons in the hidden layer. The .01 multiplier reels in the initialization of    b1    which, because biases are only added in the neuron, could technically be 0, but in a hidden layer tend to work better with a little "play" left in them.
    
    # Batch Normalization Explanation - Normalizing the batch to an approximate, but not exact, Gaussian distribution optimizes the network's ability to learn by mitigating the constant change in the distribution of layer inputs as previous layers’ parameters update during training, i.e. mitigating "internal covariate shift". By stabilizing inputs in this manner, the network ensures that when the preactivations wind up on the extremely small or large sides (where the gradient saturates and resolution is lost, i.e. the "vanishing gradient" problem, or become so unstable that they grow uncontrollably, i.e. the "exploding gradient" problem), they're instead kept in a range that preserves resolution instead of oversaturating in one direction or another, thereby allowing the network to learn effectively. The gain and bias parameters allow the network to also learn to scale and shift those normalized preactivations, moving the mean or adjusting the spread accordingly, to whatever range best captures the patterns in the data. The running mean and standard deviation are used during inference to ensure that the normalization is consistent with what was learned during training, even when processing one example at a time. This was the breakthrough in Ioffe and Szegedy's seminal 2015 paper "Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift"    https://static.googleusercontent.com/media/research.google.com/en//pubs/archive/43442.pdf    .
    batch_norm_gain_1 = torch.ones(1, num_of_neurons_in_hidden_layer) # This is the gain parameter for batch normalization, which is a learnable parameter tensor of shape (1, num_of_neurons_in_hidden_layer,). It's initialized to all ones, which means the normalized activations are not scaled initially.
    batch_norm_bias_1 = torch.zeros(1, num_of_neurons_in_hidden_layer) # This is the bias parameter for batch normalization, which is a learnable parameter tensor of shape (1, num_of_neurons_in_hidden_layer,). It's initialized to all zeros, which means no shift is applied initially.
    batch_norm_mean_running_1 = torch.zeros(1, num_of_neurons_in_hidden_layer) # This is the running mean for batch normalization, which is a non-learnable parameter tensor of shape (1, num_of_neurons_in_hidden_layer,). It's initialized to all zeros and will be updated during training to track the mean of the activations for each neuron in the hidden layer.
    batch_norm_standard_deviation_running_1 = torch.ones(1, num_of_neurons_in_hidden_layer) # This is the running standard deviation for batch normalization, which is a non-learnable parameter tensor of shape (1, num_of_neurons_in_hidden_layer,). It's initialized to all ones and will be updated during training to track the standard deviation of the activations for each neuron in the hidden layer.
    batch_norm_epsilon = 1e-5 # This is a small constant added to the denominator in the batch normalization formula to prevent division by zero when normalizing the activations. It's a common hyperparameter in batch normalization implementations.
    batch_norm_momentum = 0.001 # This is the momentum parameter for updating the running mean and standard deviation in batch normalization. It controls how much of the new batch's statistics are incorporated into the running estimates.

    W2 = torch.randn((num_of_neurons_in_hidden_layer, vocab_size), generator=g) * .01 # This is the randomly initialized (drawn from a standard normal, i.e. Gaussian, distribution) second weight matrix which, in this case, is used to train the output layer. Its first dimension is the number of neurons in the hidden layer from which it follows, and its second dimension is the number of possible outputs, i.e. the size of the vocabulary which, in the case of this character level model, is simply all possible characters, which includes the parsing character. The .01 multiplier reels in the initialization of    W2    which, as part of the last layer, could technically be 0 but to avoid getting stuck in a degenerate state is best to make a small non zero value like .01 to break symmetry and ensure the neurons maintain enough "play" to begin learning different features during training.
    b2 = torch.randn(vocab_size, generator=g) * 0 # This is the randomly initialized (drawn from a standard normal, i.e. Gaussian, distribution) bias vector for the output layer, which is added to the output of the linear transformation with    W2    before nonlinearity is applied. Its length is accordingly equal to the number of possible outputs, i.e. the size of the vocabulary which, in the case of this character level model, is simply all possible characters, which includes the parsing character. The 0 multiplier intentionally reels in the initialization of    b2    to 0 across the entire tensor because that's what works best in the last layer.

    parameter_tensors = [C, W1, b1, batch_norm_gain_1, batch_norm_bias_1, W2, b2] # This is a list of the 7 top-level learnable tensors (C, W1, b1, batch_norm_gain_1, batch_norm_bias_1, W2, b2) that compose the model.

    print('Number of parameters in untrained, freshly and randomly initialized model: ')

total_parameters = sum(p.nelement() for p in parameter_tensors) # This is the total count of all scalar values across all of those 7 tensors. The    nelement    method is a PyTorch method that returns the total number of scalars in a tensor regardless of its shape, so calling that for each top-level parameter tensor that composes the model and summing those values gives the total number of parameters that compose the model.

print (total_parameters)

In [ ]:
# Initialize training and visualization parameters.

for p in parameter_tensors: # By default, when you create a tensor (e.g., with    torch.randn    , or with    torch.tensor    , etc.), its    requires_grad    prop is False. This would mean gradients would not be tracked for those tensors...
	p.requires_grad = True # ...so explicitly setting each to True is necessary.

learning_rate_exponents = torch.linspace(-1, -1.5, 10000) # The    linspace    method in PyTorch generates a one-dimensional tensor containing a specified number of points, linearly spaced within the interval designated by the first two arguments.
learning_rates = 10 ** learning_rate_exponents # Raising 10 to the power of the    learning_rate_exponents    tensor, PyTorch performs an element-wise operation where each exponent in the    learning_rate_exponents    tensor creates a learning rate schedule that exponentially decays accordingly.

# Store the training step indices and their corresponding loss values in separate lists for the purpose of visualization with    matplotlib.pyplot    .
training_step_indices = []
loss_values = []

In [ ]:
# Train the model.

num_of_training_steps = 500000 # This is the total number of training steps to perform, which is an arbitrary hyperparameter that can be tuned for better performance. A training step consists of a forward pass, a backward pass, and an update of the model parameters.

mini_batch_size = 32 # This is the number of training examples to use with each training step. Mini-batch gradient descent uses a subset of the data for each update, which is more efficient than batch gradient descent (using the entire dataset for each update) and provides a more stable and accurate estimate of the gradients than using a single example at a time (stochastic gradient descent).

for i in range(num_of_training_steps):

	ixs = torch.randint(0, X_train.shape[0], (mini_batch_size,)) # Construct a random mini-batch for each training step by sampling    mini_batch_size    indices from the training dataset. The    torch.randint    function generates a tensor of random integers between the specified low (inclusive) and high (exclusive) values, with the shape specified in the third argument. The high boundary is set to    X_train.shape[0]    , i.e. the number of training examples. The shape is a tuple that determines the dimensions of the output tensor, which in this case, is a 1D tensor of length    mini_batch_size    .

	# Forward pass:
	embedded = C[ X_train[ixs] ] # With the list of randomly sampled indices generated on a per iteration basis, index into the training input tensor to get the list of corresponding contexts for this random mini-batch, and then with that list, index into the embedding matrix,    C    , with those contexts to get the corresponding character embeddings for each context in the random mini-batch. The resulting    embedded    tensor has a shape of (mini_batch_size, context_length, embedding_dimensions), which is a 3D tensor where each entry corresponds to the embedded representation of a character in the context for each example in the mini-batch.
	h1_preactivated = embedded.view(-1, concatenated_context_length) @ W1 + b1 # The    view    method is used to reshape the    embedded    tensor from its original shape of (mini_batch_size, context_length, embedding_dimensions) to a 2D tensor of shape (mini_batch_size, concatenated_context_length). Using -1 as its first argument to effectively maintain the mini-batch dimension, it flattens the other two dimensions by explicitly setting the second dimension to    concatenated_context_length    , which is the product of    context_length    and    embedding_dimensions    . This effectively concatenates the embeddings of the context characters into a single vector for each example in the mini-batch. This reshaped tensor is then multiplied with the first weight matrix,    W1    , then added to the bias vector,    b1    .


	batch_norm_mean_i_1 = h1_preactivated.mean(0, keepdim=True) # Calculate the mean of the preactivations across the mini-batch for each neuron in the hidden layer, resulting in a tensor of shape (1, num_of_neurons_in_hidden_layer).
	batch_norm_standard_deviation_i_1 = h1_preactivated.std(0, keepdim=True) # Calculate the standard deviation of the preactivations across the mini-batch for each neuron in the hidden layer, resulting in a tensor of shape (1, num_of_neurons_in_hidden_layer).
	h1_batch_normed = batch_norm_gain_1 * (h1_preactivated - batch_norm_mean_i_1) / (batch_norm_standard_deviation_i_1 + batch_norm_epsilon) + batch_norm_bias_1 # Normalize the preactivations by subtracting the batch mean and dividing by the batch standard deviation (plus a small epsilon for numerical stability), then scale and shift the normalized values using the learnable gain and bias parameters for batch normalization. The resulting    h1_batch_normed    tensor has the same shape as    h1_preactivated    , which is (mini_batch_size, num_of_neurons_in_hidden_layer).
	with torch.no_grad(): # Update the running mean and standard deviation for batch normalization using the momentum. This is done without tracking gradients, since these running statistics are not learnable parameters but rather are used only for normalization during inference. If you were to do this without the    torch.no_grad    context manager PyTorch would build out an entire computational graph out of these tensors because its default expectation is that    .backward    will be called on these.
		batch_norm_mean_running_1 = (1 - batch_norm_momentum) * batch_norm_mean_running_1 + batch_norm_momentum * batch_norm_mean_i_1
		batch_norm_standard_deviation_running_1 = (1 - batch_norm_momentum) * batch_norm_standard_deviation_running_1 + batch_norm_momentum * batch_norm_standard_deviation_i_1

	h1_activated = torch.tanh(h1_batch_normed) # Pass the batch normalized preactivations through the nonlinear activation function, in this case the hyperbolic tangent activation function (tanh), to produce the hidden layer of activations,    h1_activated    , which is the only hidden layer in this particular network, and for which each activation has a shape of (mini_batch_size, num_of_neurons_in_hidden_layer).
	logits = h1_activated @ W2 + b2 # The hidden layer of activations,    h1_activated    , are then multiplied with the second weight matrix,    W2    , which has a shape of (num_of_neurons_in_hidden_layer, vocab_size), and added to the bias vector,    b2    , which has a shape of (vocab_size,), to produce the output logits, which have a shape of (mini_batch_size, vocab_size). Each entry in the logits tensor corresponds to the unnormalized log probabilities of each possible next character in the vocabulary for each example in the mini-batch.

	# Normalize the logits to get probabilities:
		# counts = logits.exp() # Exponentiate the logits to get "fake counts"...
		# prob = counts / counts.sum(1, keepdims=True)# ...then normalize them into probabilities...
		# loss = - prob(torch.arrange(mini_batch_size), Y_train).log().mean() # ...then calculate the negative log likelihood loss by indexing into the probabilities with the target indices, taking the log of those probabilities, negating them, and then taking the mean across the mini-batch. Note, this is just a classification process that    torch.cross_entropy    does internally. Rolling a manual implementation of this process is not recommended because    torch.cross_entropy    does this more efficiently for the following reasons: 1) It avoids creating new tensors in memory by clustering these operations and performing them in fused kernels, which rolling one's own implementation like the above would not do, 2) The backward pass becomes more efficient because the probability calculation on which the derivative must be performed is a simpler mathematical expression, and 3)    torch.cross_entropy    is inherently more "well behaved" because it normalizes very positive numbers before they wind up too far out of range to perform    exp    on them, i.e. where they would return "inf" instead.
	loss = F.cross_entropy(logits, Y_train[ixs]) # So use PyTorch's built-in    cross_entropy    function to compute the loss directly from the logits and the target indices.

	# Backward pass:
	for p in parameter_tensors: # For each of the model's parameter tensors...
		p.grad = None # ...set the gradient to None before calling    backward()    . Because PyTorch accumulates gradients by default, new gradients would otherwise be added to the old ones from the previous iteration, which would lead to incorrect updates. Setting them to None effectively resets the gradients for the current training step, ensuring that only the gradients from the current forward pass are used for the parameter updates...
	loss.backward() # ...and finally call    backward()    on the loss to compute the gradients of the loss with respect to each of the model's parameter tensors, which are now stored in the    .grad    attribute of each tensor in the    parameter_tensors    list.

	# Update the learning:
	if os.path.exists('char_level-embed_mlp_model.pt'):
		lr = 0.01 # If a saved model file exists and is loaded, it means the model has already undergone the initial training run with the more aggressive learning rates, so use a fixed learning rate of only 0.01...
	else:
		lr = learning_rates[i] if i < 5000 else 0.035 if i < 10000 else 0.02 if i < 50000 else 0.01 if i < 100000 else .05 #  ...otherwise this is an untrained, freshly and randomly initialized model, so use this learning rate schedule for the very first training run that uses the precomputed learning rates from the    learning_rates    tensor for the first 10,000 training steps, then uses fixed thresholds in a ternary for the subsequent ranges. This approach allows for a gradual decay of the learning rate over time, which can help with convergence and prevent overshooting minima in the loss landscape.

	# Update the parameters:
	for p in parameter_tensors: # For each of the model's parameter tensors, update the parameters by subtracting the product of the learning rate and the computed gradients from the current parameter values. This effectively moves parameters in the direction that reduces the loss. The    data    attribute is used to directly modify the underlying data of the tensor without tracking this operation in the computational graph.
		p.data += -lr * p.grad

	# Track to later graph with Matplotlib:
	training_step_indices.append(i)
	loss_values.append(loss.log10().item()) # The    log10    of the loss is taken to make the loss curve more visually interpretable, especially if the loss values span several orders of magnitude during training. The    item()    method is used to extract the scalar value from the tensor, which is necessary for appending it to a standard Python list for plotting with Matplotlib.
	
	# Print the training progress:
	print(f"\rSteps: {i+1}/{num_of_training_steps} | Loss: {loss}", end="", flush=True)

In [ ]:
# Graph the loss curve.

plt.plot(training_step_indices, loss_values) # Use Matplotlib to plot the training step indices against the corresponding loss values, which have been transformed to their logarithmic scale (base 10) for better visualization of the loss curve.

In [ ]:
# Evaluate the model on the training dataset.

embedded = C[X_train] # Get the character embeddings...
h1_preactivated = embedded.view(-1, concatenated_context_length) @ W1 + b1 # ...compute the hidden layer preactivations...
h1_batch_normed = batch_norm_gain_1 * (h1_preactivated - batch_norm_mean_running_1) / (batch_norm_standard_deviation_running_1 + batch_norm_epsilon) + batch_norm_bias_1 # ...normalize the preactivations with the running batch norm statistics used for inference...
h1_activated = torch.tanh(h1_batch_normed) # ...compute the activations...
logits = h1_activated @ W2 + b2 # ...compute the output logits...
loss = F.cross_entropy(logits, Y_train) # ...compute the loss.

print(loss)

In [ ]:
# Evaluate the model on the validation dataset.

embedded = C[X_validate] # Get the character embeddings...
h1_preactivated = embedded.view(-1, concatenated_context_length) @ W1 + b1 # ...compute the hidden layer preactivations...
h1_batch_normed = batch_norm_gain_1 * (h1_preactivated - batch_norm_mean_running_1) / (batch_norm_standard_deviation_running_1 + batch_norm_epsilon) + batch_norm_bias_1 # ...normalize the preactivations with the running batch norm statistics used for inference...
h1_activated = torch.tanh(h1_batch_normed) # ...compute the activations...
logits = h1_activated @ W2 + b2 # ...compute the output logits...
loss = F.cross_entropy(logits, Y_validate) # ...compute the loss.

print(loss)

In [ ]:
# Evaluate the model on the testing dataset.

embedded = C[X_test] # Get the character embeddings...
h1_preactivated = embedded.view(-1, concatenated_context_length) @ W1 + b1 # ...compute the hidden layer preactivations...
h1_batch_normed = batch_norm_gain_1 * (h1_preactivated - batch_norm_mean_running_1) / (batch_norm_standard_deviation_running_1 + batch_norm_epsilon) + batch_norm_bias_1 # ...normalize the preactivations with the running batch norm statistics used for inference...
h1_activated = torch.tanh(h1_batch_normed) # ...compute the activations...
logits = h1_activated @ W2 + b2 # ...compute the output logits...
loss = F.cross_entropy(logits, Y_test) # ...compute the loss.

print(loss)

# Note, if the training process is successful, the training loss should be the lowest, followed by the validation loss, and then the test loss, which should be the highest. This is because the model is trained to minimize the training loss, and while it should generalize well to unseen data (validation and test sets), it typically performs best on the data it was actually trained on. The validation loss is used to tune hyperparameters and prevent overfitting, while the test loss provides an unbiased evaluation of the final model's performance on completely unseen data.

In [ ]:
# Save model parameters.

# If loss evaluations above improved the model, run this cell to save the updates.
torch.save({
    'C': C,
    'W1': W1,
    'b1': b1,
    'batch_norm_gain_1': batch_norm_gain_1,
    'batch_norm_bias_1': batch_norm_bias_1,
    'batch_norm_mean_running_1': batch_norm_mean_running_1,
    'batch_norm_standard_deviation_running_1': batch_norm_standard_deviation_running_1,
    'batch_norm_epsilon': batch_norm_epsilon,
    'batch_norm_momentum': batch_norm_momentum,
    'W2': W2,
    'b2': b2
}, 'char_level-embed_mlp_model.pt')

In [ ]:
# Visualize the character embeddings.

# This only visualizes the first two dimensions of the embedding matrix,    C    , for all characters. This is a common practice to visualize high-dimensional embeddings by projecting them down to 2D space. Each point in the scatter plot corresponds to a character in the vocabulary, and its position is determined by the values of the first two dimensions of its embedding vector.
plt.figure(figsize=(8,8))
plt.scatter(C[:,0].detach().numpy(), C[:,1].detach().numpy(), s=200)
for i in range(C.shape[0]):
    plt.text(C[i,0].item(), C[i,1].item(), index_to_string[i], ha="center", va="center", color='white')
plt.title('Character Embeddings of only the First 2 Dimensions')
plt.grid('minor')


#  This visualizes all dimensions of the embedding matrix,    C    , for all characters using UMAP.
umap_model = umap.UMAP(n_components=2, random_state=42)
C_umap = umap_model.fit_transform(C.detach().numpy())

plt.figure(figsize=(8,8))
plt.scatter(C_umap[:,0], C_umap[:,1], s=200)
for i in range(C_umap.shape[0]):
    plt.text(C_umap[i,0], C_umap[i,1], index_to_string[i], ha="center", va="center", color='white')
plt.title(f'Character Embeddings of all {C.shape[1]} Dimensions Reduced with UMAP')
plt.grid('minor')

In [ ]:
# Autoregressively sample from the model.

random_pytorch_seed = 321 # Define an arbitrary seed to pass to PyTorch's random number generator. This is purely to be able to reproduce the same values when rerunning the code, effectively and ironically making the random number generator nonrandom, but which is important for debugging and consistency of results. Without setting a seed, the initialization of the model parameters would yield different results each time the code ran. This could technically be any number.
g = torch.Generator().manual_seed(random_pytorch_seed)

num_to_generate = 20 # This is just the number of new words to generate from the trained model.

for _ in range(num_to_generate): # Use the trained model to generate    num_to_generate    new words by sampling from the model's learned probability distribution over the next character given a context of previous characters. This is done by repeatedly feeding the model's output back into itself as input until the parsing character is generated, which indicates the end of a word. In the consumer sense, this is effectively running inference on the model in an autoregressive loop.
    
    generated_indices = [] # Initialize an empty list to store the generated character indices.
    context = [ string_to_index[parsing_character] ] * context_length # Same as before when initializing the datasets, initialize the context list of repeating indices corresponding to the parsing character, with the length of the list equal to the    context_length    .
    
    while True: # while the parsing character has not yet been generated, keep generating characters by feeding the model's output back into itself as input.
        embedded = C[ torch.tensor([context]) ] # Index into the embedding matrix,    C    , with the current context to get the corresponding character embeddings for the current context. The resulting    embedded    tensor has a shape of (1, context_length, embedding_dimensions), which is a 3D tensor corresponding to the embedded representation of the characters across the current context.
        h1_preactivated = embedded.view(1, -1) @ W1 + b1 # Flatten the embedded tensor to shape (1, concatenated_context_length) and compute the hidden layer preactivations,    h1_preactivated    , by multiplying with    W1    , and adding    b1    . Because the reshaped    embedded    tensor has a first dimension of 1, PyTorch's broadcasting rules allow for matrix multiplication to be performed with    W1    . When you consider what's happening, this makes sense because flattening the embeddings concatenates them into a 2D tensor for which one of the dimensions is 1, and each one of those 2D tensors is broadcast to each of the neurons in    W1    , which of course shares with    b1    the same second dimension, which is the number of neurons in the hidden layer. Hence the resulting   h1_preactivated    tensor has a shape of (1, num_of_neurons_in_hidden_layer), which corresponds to the preactivations of the hidden layer.
        h1_batch_normed = batch_norm_gain_1 * (h1_preactivated - batch_norm_mean_running_1) / (batch_norm_standard_deviation_running_1 + batch_norm_epsilon) + batch_norm_bias_1 # Normalize the hidden layer preactivations using the running mean and standard deviation for batch normalization, then scale and shift the normalized values using the learnable gain and bias parameters for batch normalization. The resulting    h1_batch_normed    tensor has the same shape as    h1_preactivated    , which is (1, num_of_neurons_in_hidden_layer).
        h1_activated = torch.tanh(h1_batch_normed) # Pass the batch normalized preactivation through the nonlinear activation function, in this case the hyperbolic tangent activation function (tanh), to produce the hidden layer of activations.
        logits = h1_activated @ W2 + b2 # Compute the output logits by multiplying the hidden layer activations,    h1_activated    , with    W2    and adding    b2    , resulting in a tensor of shape (1, vocab_size) that contains the unnormalized log probabilities of each possible next character in the vocabulary given the current context. Because the    h1_activated    tensor has a first dimension of 1, PyTorch's broadcasting rules allow for matrix multiplication to be performed with    W2    . When you consider what's happening, this makes sense because each of the hidden layer activations is shared across all possible output characters in    W2    , which of course shares with    b1    the same second dimension. Hence the resulting    logits    tensor has a shape of (1, vocab_size), which corresponds to the unnormalized log probabilities of each possible next character in the vocabulary given the current context.
        probs = F.softmax(logits, dim=1) # Normalize the logits to get probabilities by applying the softmax function along the appropriate dimension, which converts the unnormalized log probabilities into a valid probability distribution over the next character given the current context.
        ix = torch.multinomial(probs, num_samples=1, generator=g).item() # Sample the next character index from the probability distribution using    torch.multinomial    , which randomly samples from the distribution defined by the probabilities. The    num_samples=1    argument specifies that we want to sample one index. The    .item()    method is used to extract the sampled index as a standard Python integer rather than a PyTorch tensor.
        context = context[1:] + [ix] # Update the context by removing the first element and appending the newly generated character index, effectively sliding the context window forward by one character to prepare it for the next iteration of generation.
        generated_indices.append(ix) # Append the newly generated character index to the list of generated indices, which will be used to construct the generated word once the parsing character is generated.
        if ix == string_to_index[parsing_character]: break # If the generated index is 0, which corresponds to the parsing character, break the loop as this indicates the end of the generated word.

    print(''.join(index_to_string[i] for i in generated_indices)) # After the loop breaks, use the list of generated character indices to construct the generated word by mapping each index back to its corresponding character using the    index_to_string    dictionary, and then joining those characters together into a single string to print as the generated word.